## LOAD DATASET

In [60]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

In [61]:
df.shape

(50000, 2)

In [62]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [63]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [64]:
df.drop_duplicates(inplace=True)

In [65]:
df.shape

(49582, 2)

## Pre-processing

### 1. Converting to lowercase

In [66]:
df["review"] = df["review"].str.lower()

### 2. Removing URls

In [68]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)  # (pattern, replace, text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [70]:
def remove_punctuations(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text) 
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing the HTMLTags

In [72]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text) 
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing stopwords

In [83]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Deepak\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Deepak\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Deepak\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [85]:
from nltk.tokenize import word_tokenize
from nltk.corpus  import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
            
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [86]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [87]:
from nltk.stem import PorterStemmer

def stemming(text):
    tokens = word_tokenize(text)
    ps = PorterStemmer()
    stemmed_words = []

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] =  df["review"].apply(stemming)

In [88]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [89]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [90]:
y = df["sentiment"]

In [91]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [94]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [95]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset and Dataloaders

In [97]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)

In [98]:
X_test.shape

(9917, 5000)

In [99]:
X_train.shape

(39665, 5000)

In [101]:
from torch.utils.data import TensorDataset, DataLoader
import torch

In [104]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [105]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [106]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build the Model

In [112]:
import torch.nn as nn
import torch.optim as optim

In [113]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=1, batch_first=True)

        # fc layers
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (nums of layers, batch size, self.hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all timesetps => (batch, seq_len, hiddden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [114]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training and Evaluating

In [117]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        outputs = model(Xb) # (batch_size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probilibity

        loss = criterion(outputs, yb) # compute the loss
        loss.backward() # back propagation
        optimizer.step() # weight update

    print(f" epoch = {epoch+1}/{epochs} and loss = {loss}")

 epoch = 1/10 and loss = 0.3685426414012909
 epoch = 2/10 and loss = 0.28813275694847107
 epoch = 3/10 and loss = 0.17377044260501862
 epoch = 4/10 and loss = 0.4315343499183655
 epoch = 5/10 and loss = 0.18478459119796753
 epoch = 6/10 and loss = 0.12444359064102173
 epoch = 7/10 and loss = 0.18581916391849518
 epoch = 8/10 and loss = 0.2151874303817749
 epoch = 9/10 and loss = 0.1470990926027298
 epoch = 10/10 and loss = 0.13869236409664154


In [120]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float() # (batch_size,) => probilibity

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f" accuracy = {correct_vals/total_vals*100}")

 accuracy = 85.36855904003227
